In [ ]:
fname = 'conipfv_h11=9_polys[-250toNone]_dilation=100_numpvecs=20000_time2026-01-30_11-02-43'

# Imports

CYTools

In [ ]:
from cytools import Polytope

Local to PFV repo

In [ ]:
import sys; sys.path.append('..')
from src import diagnostics, cydata

External

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Reconstruct the PFVs

Read from the data file

In [ ]:
with open('PFVSTRINGS-'+fname+'.txt', 'r') as f:
    pfv_strs = ['']
    
    for l in f:
        if l=='\n':
            pfv_strs.append('')
        else:
            pfv_strs[-1] = pfv_strs[-1] + l
pfv_strs = pfv_strs[:-1]

Clean the strings, trimming extra data appended to the PFV

In [ ]:
import re

all_numLG = []
all_numBG = []

for i,pfv_str in enumerate(pfv_strs):
    split = pfv_str.split('\n')

    # extract the substring corresponding to the PFV
    for j,l in enumerate(split):
        if l[:4] == '#Q =':
            break
    
    pfv_bit = '\n'.join(split[:j+1])

    # other data
    SG_str = split[j+1:-1][0]
    m = re.search(r'\((\d+)\s*,\s*(\d+)\)\s*$', SG_str)
    if m:
        numLG, numBG = map(int, m.groups())

    all_numLG.append(numLG)
    all_numBG.append(numBG)

    # overwrite existing data
    pfv_strs[i] = pfv_bit

Make into formal PFV objects

In [ ]:
pfvs = [diagnostics.PFV.from_str(str_) for str_ in tqdm(pfv_strs)]

Assign GV info

In [ ]:
verts_heights_to_gvs = {}

for pfv in tqdm(pfvs):
    verts_safe = tuple([tuple(v) for v in pfv.verts])
    heights_safe = tuple(pfv.heights)
    key = (verts_safe,heights_safe)

    if key in verts_heights_to_gvs:
        pfv.gvs = verts_heights_to_gvs[key]
    else:
        cy = pfv.cy
        verts_heights_to_gvs[key] = cy.compute_gvs(max_deg=10).coo
        pfv.gvs = verts_heights_to_gvs[key]

# Read data from the PFVs

Load SG database to check if these have LG SGs

In [ ]:
import duckdb

con = duckdb.connect('kklt_db.duckdb')
tmp = con.execute("SHOW ALL TABLES").fetchdf()
df_conis = con.execute("SELECT * FROM conifolds").fetchdf()
df_sgs   = con.execute("SELECT * FROM starting_guesses").fetchdf()
con.close()

In [ ]:
import sys; sys.path.append('../../cornell-dev')
from lib.cytools_ext import polytope_ext
from projects.kklt.kklt_lib import kklt_conifolds

In [ ]:
W0    = []
align = []
gsM   = []
gsM2  = []
M0    = []
Kprime= []
numLG = []
numBG = []

for pfv,nLG,nBG in tqdm(zip(pfvs,all_numLG,all_numBG)):
    pfv_w0 = pfv.W0()
    if pfv_w0 is np.nan:
        W0.append(np.nan)
        align.append(np.nan)
        gsM.append(np.nan)
        gsM2.append(np.nan)
        M0.append(pfv.M[0])
        Kprime.append(pfv.Kprime)
        numLG.append(nLG)
        numBG.append(nBG)
        continue

    # save the data
    # -------------
    W0.append(pfv.W0())
    align.append(pfv.align)
    gsM.append(pfv.gsM)
    gsM2.append(pfv.gsM*pfv.M[0])
    M0.append(pfv.M[0])
    Kprime.append(pfv.Kprime)
    numLG.append(nLG)
    numBG.append(nBG)

df = pd.DataFrame({
    'W0':W0,
    'align':align,
    'gsM':gsM,
    'gsM2':gsM2,
    'M0':M0,
    'Kprime':Kprime,
    'numLG':numLG,
    'numBG':numBG
})

# Check that th PFV has an (LG) SG

# Plot the data!

In [ ]:
s = 0.5

df_tmp = df[df['gsM']>5]

df_plot = df_tmp[df_tmp['numLG']>0]
plt.scatter(df_plot['W0'], df_plot['align'], c=df_plot['gsM'], s=s, marker='o')

df_plot = df_tmp[df_tmp['numLG']==00]
plt.scatter(df_plot['W0'], df_plot['align'], c=df_plot['gsM'], s=s, marker='x')
plt.colorbar(label='gsM')

# limits
#plt.xlim([8e-4,1])
#plt.ylim([0.1,1.2])
#plt.clim([1,3])
if 0:
    plt.xlim([8e-4,1])
    plt.ylim([10**1, 10**10])

# scales
plt.xscale('log')
plt.yscale('log')

# labels
plt.title(fname)
plt.xlabel('W0')
plt.ylabel('align')

In [ ]:
df[(df['gsM']>2) & (df['gsM']<3) & (df['align']<=10)]

In [ ]:
pfvs[14311].diagnostics()

In [ ]:
df[(df['align']>0.1) & (df['align']<1.2)]

In [ ]:
interesting[3]

In [ ]:
interesting = [1370, 1542, 1628, 1721, 1728, 2259]

In [ ]:
pfv = diagnostics.PFV.from_str(str(pfvs[interesting[0]]))

In [ ]:
pfv = pfvs[interesting[5]].diagnostics()

In [ ]:
for i in interesting[:1]:
    pfv = pfvs[i]

    # recompute the GVs at max_deg=15
    verts_safe = tuple([tuple(v) for v in pfv._cy.verts])
    heights_safe = tuple(pfv._cy.heights)

    cy = Polytope(verts_safe).triangulate(heights=heights_safe).cy()
    pfv.gvs = cy.compute_gvs(max_deg=1).coo

    print(pfv.harvest()['alignment'], pfv.align)
    print()

In [ ]:
Tallowed = 100 # hrs
max_dilation = 20 * (Tallowed/2)**0.25

In [ ]:
max_dilation

In [ ]:
df.iloc[1370]

In [ ]:
pfvs[1370].diagnostics()

# Scratch